# Write ATST example 2

Build a two-readout ATST package with linked metadata and readings.

## Setup

Import the ATST block classes, multi-readout writer, CLARIOstar reader, and data utilities.

In [1]:
from pathlib import Path

import pandas as pd

from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    ReadoutIds,
    Study,
)
from ATST import ClariostarReader


## Input data

Read the GFP and OD600 exports and their shared plate layout.

In [2]:
reader = ClariostarReader()
gfp_output = reader.read(
    Path("raw_outputs/simulated_clariostar_gfp_output.csv").read_bytes(),
    filename="simulated_clariostar_gfp_output.csv",
)
od600_output = reader.read(
    Path("raw_outputs/simulated_clariostar_od600_output.csv").read_bytes(),
    filename="simulated_clariostar_od600_output.csv",
)
layout_data = pd.read_csv("example_2_layout.tsv", sep="\t", keep_default_na=False)

od600_output.readings.head()


,Time,A1,B1,C1,A2,B2,C2,A3,B3,C3,...,C8,A9,B9,C9,A10,B10,C10,A11,B11,C11
0,0,0.146,0.198,0.149,0.151,0.155,0.182,0.167,0.174,0.183,...,0.175,0.148,0.16,0.175,0.174,0.166,0.18,0.17,0.172,0.189
1,40,0.175,0.187,0.181,0.16,0.154,0.184,0.19,0.192,0.188,...,0.184,0.167,0.176,0.175,0.15,0.154,0.194,0.165,0.168,0.166
2,80,0.204,0.22,0.232,0.157,0.15,0.223,0.217,0.222,0.218,...,0.163,0.163,0.172,0.155,0.161,0.171,0.182,0.23,0.211,0.213
3,120,0.29,0.256,0.254,0.164,0.145,0.263,0.295,0.307,0.315,...,0.175,0.168,0.171,0.17,0.157,0.161,0.185,0.267,0.276,0.27
4,160,0.338,0.359,0.376,0.181,0.201,0.374,0.352,0.374,0.374,...,0.186,0.169,0.163,0.164,0.149,0.185,0.189,0.378,0.374,0.338


## FILE_INFO

`FILE_INFO` identifies the output file and ATST format. The file name, format, version, creation date, delimiter, and encoding are required.

In [3]:
file_info = FileMetadata(data={
    "file_name": "example_2.atst.txt",
    "format": "ATST",
    "format_version": "0.1",
    "created_on": "2026-01-01",
    "field_delimiter": "TAB",
    "encoding": "UTF-8",
})


## STUDY

`STUDY` identifies and describes the experiment. A non-empty title and study ID are required.

In [4]:
study = Study(data={
    "title": "Example 2 simulated multi-readout assay",
    "study_id": "ATST_EXAMPLE_2",
    "description": "Simulated OD600 and GFP readouts with linked metadata and readings.",
})


## READOUT_IDS

`READOUT_IDS` lists the unique IDs used to connect each readout-specific block. It is required for a multi-readout ATST file.

In [5]:
readout_ids = ReadoutIds(data=pd.DataFrame({"readout_id": ["GFP", "OD600"]}))


## METADATA: GFP

This `METADATA` block records acquisition details for the GFP readout.

In [6]:
gfp_metadata = Metadata(data=gfp_output.metadata)


## ASSAY: GFP

This `ASSAY` block defines the signal, unit, time unit, and plate format for GFP.

In [7]:
gfp_assay = Assay(data=gfp_output.assay)


## METADATA: OD600

This `METADATA` block records acquisition details for the OD600 readout.

In [8]:
od600_metadata = Metadata(data=od600_output.metadata)


## ASSAY: OD600

This `ASSAY` block defines the signal, unit, time unit, and plate format for OD600.

In [9]:
od600_assay = Assay(data=od600_output.assay)


## ENTITIES

`ENTITIES` defines the biological records referenced by the layout. Each table declares a primary key with unique, non-empty values. These tables should be generated before constructing the final ATST file.

In [10]:
phages = EntityTable(
    name="ENTITIES",
    table_name="PHAGES",
    pk="PHAGE_ID",
    data=pd.read_csv("entities/PHAGES.csv", keep_default_na=False),
)
isolates = EntityTable(
    name="ENTITIES",
    table_name="ISOLATES",
    pk="ISOLATE_ID",
    data=pd.read_csv("entities/ISOLATES.csv", keep_default_na=False),
)
entities = Entities(tables={"PHAGES": phages, "ISOLATES": isolates})


## LAYOUT: GFP

The GFP layout maps wells to the shared entity records.

In [11]:
gfp_layout = Layout(data=layout_data)


## READINGS: GFP

The GFP readings use the well names defined by its layout.

In [12]:
gfp_readings = Readings(data=gfp_output.readings)


## LAYOUT: OD600

The OD600 layout maps wells to the shared entity records.

In [13]:
od600_layout = Layout(data=layout_data)


## READINGS: OD600

The OD600 readings use the well names defined by its layout.

In [14]:
od600_readings = Readings(data=od600_output.readings)


## Assemble and validate

Create one ATST object per readout, combine them, and validate the complete package.

In [15]:
gfp = ATSTFile(
    file_info=file_info, study=study, readout_ids=readout_ids,
    metadata=gfp_metadata, assay=gfp_assay, entities=entities,
    layout=gfp_layout, readings=gfp_readings,
)
od600 = ATSTFile(
    file_info=file_info, study=study, readout_ids=readout_ids,
    metadata=od600_metadata, assay=od600_assay, entities=entities,
    layout=od600_layout, readings=od600_readings,
)
atst = MultiReadoutATST(
    file_info, study, readout_ids, {"GFP": gfp, "OD600": od600}, entities
)
validate_atst(atst)


## Write the ATST package

Keep assay, entities, and layout inline; write metadata and readings as linked TSV files.

In [16]:
write_multi_readout_atst(
    atst, ".", linked_files=True, linked_blocks={"METADATA", "READINGS"}
)


PosixPath('example_2.atst.txt')